In [ ]:
import os
import json
import csv
from PIL import Image
import pandas as pd


In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
# Set these paths before running. All paths are relative to this notebook (utils/).

# Folder containing the JSON sidecar files produced by the segmentation script.
# Pick the output folder that matches the pipeline you ran:
#   Grounding DINO + SAM 2  ->  "../notebooks/seg_experiment/groundingdino_outputs"
#   Florence-2  + SAM 2     ->  "../notebooks/seg_experiment/florence_outputs"
#   SAM 2 automatic          ->  "../notebooks/seg_experiment/sam2"
JSON_FOLDER       = "../notebooks/seg_experiment/groundingdino_outputs"

# Folder containing the original input images (filenames must match the JSON names).
IMAGE_FOLDER      = "../notebooks/data"

# Raw field measurements CSV.
# Required columns: photo, length (cm), sensor_width (mm), focal_length (mm)
# Optional column : actual_dbh (cm)  — enables MAE validation in Step 8
FIELD_CSV         = "dbh_csv.csv"

# Intermediate and final CSV paths written by this notebook.
CSV_PIXEL_WIDTHS  = "dbh_pixel_widths.csv"           # Step 1 output
CSV_WITH_DIMS     = "dbh_pixel_widths_with_dims.csv"  # Step 2 output
CSV_FINAL_MERGED  = "final_output.csv"                # Step 7 output
CSV_ESTIMATED_DBH = "estimated_dbh.csv"               # Step 8 output


In [ ]:
# Step 1 — Extract DBH pixel widths from all JSON sidecar files
#
# Uses diameter_px (Euclidean length of the PCA-perpendicular line) — accurate
# for both upright and tilted trunks. Falls back to diameter_line_coords for
# legacy JSON files that pre-date the diameter_px field.
#
# Class matching uses substring check ("trunk" in class) to be consistent
# with the segmentation scripts, which also filter on substring.

rows = []
skipped = []

for file_name in sorted(os.listdir(JSON_FOLDER)):
    if not file_name.endswith('.json'):
        continue

    file_path = os.path.join(JSON_FOLDER, file_name)
    image_name = file_name[:-5] + '.jpg'  # strip .json, add .jpg

    with open(file_path, 'r') as f:
        data = json.load(f)

    found = False
    for obj in data:
        # Match the same way the segmentation scripts do: substring check
        if 'trunk' not in obj.get('class', '').lower():
            continue

        # Prefer diameter_px (Euclidean distance — correct for tilted trunks)
        if 'diameter_px' in obj:
            width = obj['diameter_px']
        # Fallback: horizontal projection from line endpoint coords
        # Handles legacy JSON files that predate the diameter_px field
        else:
            coords = obj.get('diameter_line_coords') or obj.get('lowest_point_line_coords')
            if coords is None:
                continue
            width = coords['right'][0] - coords['left'][0]

        trunk_angle = obj.get('trunk_angle_deg', None)
        rows.append([image_name, width, trunk_angle])
        found = True
        break  # one trunk measurement per image

    if not found:
        skipped.append(image_name)

with open(CSV_PIXEL_WIDTHS, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['name', 'dbh_width', 'trunk_angle_deg'])
    writer.writerows(rows)

print(f'Extracted {len(rows)} measurements -> {CSV_PIXEL_WIDTHS}')
if skipped:
    print(f'WARNING: {len(skipped)} file(s) had no trunk detection: {skipped}')


In [ ]:
# Step 2 — Append image pixel dimensions to the pixel-width CSV.

# image_width is required by the metric conversion formula.



rows = []



with open(CSV_PIXEL_WIDTHS, 'r') as f:

    reader = csv.DictReader(f)

    for row in reader:

        image_path = os.path.join(IMAGE_FOLDER, row['name'])

        img_width, img_height = None, None

        if os.path.exists(image_path):

            with Image.open(image_path) as img:

                img_width, img_height = img.size  # (width, height)

        else:

            print(f"  WARNING: image not found: {image_path}")

        row['image_width'] = img_width

        row['image_height'] = img_height

        rows.append(row)



with open(CSV_WITH_DIMS, 'w', newline='') as f:

    fieldnames = list(rows[0].keys())

    writer = csv.DictWriter(f, fieldnames=fieldnames)

    writer.writeheader()

    writer.writerows(rows)



print(f'Updated CSV with image dimensions -> {CSV_WITH_DIMS}')


In [ ]:
# Step 3 — Load both CSVs into DataFrames for merging.

# pixel_df  : pixel widths + image dimensions from Steps 1-2

# field_df  : raw field measurements (actual DBH, camera distance, intrinsics)



pixel_df = pd.read_csv(CSV_WITH_DIMS)

field_df = pd.read_csv(FIELD_CSV)



print('Pixel CSV shape:', pixel_df.shape)

print('Field CSV shape:', field_df.shape)

print('\nPixel CSV columns:', list(pixel_df.columns))

print('Field CSV columns:', list(field_df.columns))


In [ ]:
# Step 4 — Align column names and merge on the image filename.

# The pixel CSV uses 'name'; the field CSV must use 'photo'.



pixel_df.rename(columns={'name': 'photo'}, inplace=True)



# Left join: keeps all field records even if segmentation failed for that image

merge = pd.merge(

    field_df,

    pixel_df,

    on='photo',

    how='left',

    indicator=True,

    suffixes=('_field', '_pixel'),

    validate='1:1'

)



print(f'Total rows: {len(merge)}')

print(merge['_merge'].value_counts())


In [ ]:
# Step 5 — Inspect unmatched rows.

# 'left_only' = image in field CSV but segmentation produced no trunk detection.

# Review these images individually before dropping them.



unmatched = merge[merge['_merge'] == 'left_only']

if len(unmatched) > 0:

    print(f'WARNING: {len(unmatched)} image(s) have no segmentation result:')

    print(unmatched[['photo']].to_string(index=False))

else:

    print('All field records matched a segmentation output.')


In [ ]:
# Step 6 — Preview matched rows.

merge[merge['_merge'] == 'both']


In [ ]:
# Step 7 — Keep only matched rows and save.

matched = merge[merge['_merge'] == 'both'].drop(columns=['_merge'])

matched.to_csv(CSV_FINAL_MERGED, index=False)

print(f'Saved {len(matched)} matched rows -> {CSV_FINAL_MERGED}')

print('Columns:', list(matched.columns))


In [ ]:
# Step 8 — Convert pixel DBH to real-world centimetres.
# Formula (pinhole camera model):
#   W_mm = (dbh_width * sensor_width * D_mm) / (image_width * focal_length)
#   DBH_cm = W_mm / 10
# where D_mm = length * 10  (length column is in cm)
#
# Required columns in final_output.csv:
#   dbh_width    : trunk width in pixels
#   sensor_width : camera sensor width in mm
#   length       : camera-to-trunk distance in cm
#   image_width  : full image width in pixels
#   focal_length : camera focal length in mm

df = pd.read_csv(CSV_FINAL_MERGED)

D_mm = df['length'] * 10  # cm -> mm
W_mm = (df['dbh_width'] * df['sensor_width'] * D_mm) / (df['image_width'] * df['focal_length'])
df['estimated_dbh'] = (W_mm / 10).round(4)  # mm -> cm

df.to_csv(CSV_ESTIMATED_DBH, index=False)
print(f'Saved results -> {CSV_ESTIMATED_DBH}')
print()

# Validation summary (requires actual_dbh column in the field CSV)
if 'actual_dbh' in df.columns:
    df['error_cm']  = (df['estimated_dbh'] - df['actual_dbh']).round(4)
    df['error_pct'] = ((df['error_cm'] / df['actual_dbh']) * 100).round(2)
    print(df[['photo', 'actual_dbh', 'estimated_dbh', 'error_cm', 'error_pct']].to_string(index=False))
    print(f'\nMean absolute error            : {df["error_cm"].abs().mean():.4f} cm')
    print(f'Mean absolute percentage error : {df["error_pct"].abs().mean():.2f} %')
else:
    print(df[['photo', 'estimated_dbh']].to_string(index=False))
